# Multicomponent X-ray attenuation: reproducible examples

This notebook calls the same engine as the graphical workbench. Start Jupyter in the project folder with the project's Python environment. It needs the packages in `requirements.txt`; Jupyter itself is optional and installed separately.

The independent-atom, narrow-beam model excludes secondary-photon transport and near-edge chemical structure. `1-T` means removed primary photons, not absorbed energy. Read `docs/SCIENTIFIC_METHODS.md` before interpreting results. Densities here are illustrative; substitute measured values.


In [ ]:
import copy
import json
import sys
from pathlib import Path

import numpy as np

project_dir = Path.cwd()
if not (project_dir / 'xray_workbench').exists():
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from xray_workbench.physics import calculate  # noqa: E402 - after the project path is prepared

project = json.loads((project_dir / 'examples' / 'borosilicate-multilayer.json').read_text())
configuration = project['configuration']
result = calculate(configuration)
result['reference']

## Inspect mass fractions and reference coefficients

Formula-unit mole proportions are converted using molecular masses calculated from stoichiometry. The mixture coefficient is the mass-weighted sum of component coefficients, with each mass fraction used once.


In [ ]:
for layer in result['layers']:
    print(layer['name'])
    print('Effective density (g/cm3):', layer['density_g_cm3'])
    print('Mass fractions:', [(c['formula'], c['mass_fraction']) for c in layer['components']])
    print('Reference outputs:', layer['reference'])


## Layer invariance and thickness design

Primary transmission is unchanged by reordering layers because optical depths add. This is not true for a general secondary-radiation transport calculation. The design multiplier scales every normal layer thickness to meet the monoenergetic target.


In [ ]:
reversed_config = copy.deepcopy(configuration)
reversed_config['layers'].reverse()
reversed_config['uncertainty']['enabled'] = False
assert np.allclose(calculate(reversed_config)['transmission'], result['transmission'])

scaled = copy.deepcopy(configuration)
scale = result['reference']['target_thickness_scale']
for layer in scaled['layers']:
    layer['thickness_mm'] *= scale
scaled['uncertainty']['enabled'] = False
target_result = calculate(scaled)
assert np.isclose(target_result['reference']['transmission'], configuration['target_transmission'])
print('Thickness multiplier:', scale)
print('Achieved transmission:', target_result['reference']['transmission'])


## Imported spectrum and uncertainty

Spectrum weights are integrated photon fluence per discrete bin. The sample spectrum is illustrative. The uncertainty interval below propagates only independent density and thickness uncertainties; it does not quantify database or model uncertainty.


In [ ]:
spectrum = result['spectrum']
print('Photon-weighted transmission:', spectrum['transmission'])
print('Mean energy in / out (keV):', spectrum['mean_energy_in_keV'], spectrum['mean_energy_out_keV'])
print('Reference uncertainty:', result['uncertainty']['reference'])

energy_config = copy.deepcopy(configuration)
energy_config['spectrum']['weighting'] = 'energy'
energy_config['uncertainty']['enabled'] = False
print('Energy-weighted transmission:', calculate(energy_config)['spectrum']['transmission'])


## Programmatic sweep

This example varies the glass normal thickness while retaining the aluminium support. Use the GUI to export interactive-plot data or SVG; this notebook keeps the calculations independent of a plotting dependency.


In [ ]:
thicknesses_mm = np.linspace(0, 5, 11)
transmissions = []
for thickness in thicknesses_mm:
    trial = copy.deepcopy(configuration)
    trial['layers'][0]['thickness_mm'] = float(thickness)
    trial['uncertainty']['enabled'] = False
    trial['spectrum'] = None
    transmissions.append(calculate(trial)['reference']['transmission'])
list(zip(thicknesses_mm.tolist(), transmissions, strict=True))